**Step 1: Create a sample.txt file to work with**

In [145]:
with open ("sample.txt","w") as f:
    f.write("Hello Everyone!!! I'm Tejaswi\n")
    f.write("Homework for week1\n")
    

**Step 2: Define read_file and get_content_length tools**

In [146]:
!pip install langchain langchain-ollama --quiet

In [147]:
from langchain_core.tools import tool
@tool
def read_file(file_path:str)->str:
    """ Reads the content of the file """
    with open(file_path,"r") as f:
        file_content=f.read()
    return file_content
    
@tool
def get_content_length(content:str)->int:
    """ Return the number of characters in the provided text content.
        if it is a file Use this after reading file content
    """
    return len(content)

print('Tool name   :', read_file.name)
print('Description :', read_file.description)
print('Schema      :', read_file.args)

    

Tool name   : read_file
Description : Reads the content of the file
Schema      : {'file_path': {'title': 'File Path', 'type': 'string'}}


In [148]:
result = read_file.invoke({"file_path": "sample.txt"})
print(result)

Content_length = get_content_length.invoke({'content': result })
print(Content_length)

Hello Everyone!!! I'm Tejaswi
Homework for week1

49


**Step 3: Bind tools to the LLM**

In [149]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model='llama3.2')
tools = [read_file,get_content_length]

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke('what is the length of name Tejaswi?')
print('Response   :', response.content)
print('Tool calls :', response.tool_calls)

Response   : 
Tool calls : [{'name': 'get_content_length', 'args': {'content': 'Tejaswi'}, 'id': '0e28fdbc-1004-46e4-adbe-562c1936419e', 'type': 'tool_call'}]


**Step 4: Build the agent loop- Model calls tools automatically**

In [152]:
from langchain_core.messages import HumanMessage, ToolMessage

tool_map = {'read_file': read_file, 'get_content_length': get_content_length}

# Step 1 — ask the model
messages = [HumanMessage(content='Read the contents in sample.txt file and return Total number of characters in the text file.')]

response = llm_with_tools.invoke(messages)

messages.append(response)

# Step 2 — run the tool the model asked for
for tool_call in response.tool_calls:
    tool_fn = tool_map[tool_call['name']]  #tool_fn get the name of the tool model asked for
    tool_result = tool_fn.invoke(tool_call['args'])
    messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_call['id']))


# Step 3 — send result back to the model for a final answer
final = llm_with_tools.invoke(messages)
print('Final answer:', final.content)

Final answer: Based on the file contents, the total number of characters in the text file is: 28


In [153]:
questions = [
    'What is the length of words in sample.txt?',
    'What are the contents in sample.txt?'
]

for question in questions:
    response = llm_with_tools.invoke(question)
    if response.tool_calls:
        call = response.tool_calls[0]
        print(f'Q: {question}')
        print(f'   -> Model chose tool: {call["name"]} with args {call["args"]}')
    else:
        print(f'Q: {question}')
        print(f'   -> Direct answer: {response.content}')
    print()

Q: What is the length of words in sample.txt?
   -> Model chose tool: get_content_length with args {'content': 'read_file', 'file_path': 'sample.txt'}

Q: What are the contents in sample.txt?
   -> Model chose tool: read_file with args {'file_path': 'sample.txt'}

